In [ ]:
# ELEC4240: image-generation-derived perception. Runtime check only; no training.
import json, platform, torch
from pathlib import Path
assert torch.cuda.is_available(), 'GPU runtime required'
result = {'stage':'colab_connection', 'gpu':torch.cuda.get_device_name(), 'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30, 'torch':torch.__version__, 'python':platform.python_version(), 'training_started':False}
Path('/content/generative_ris_v11').mkdir(exist_ok=True)
Path('/content/generative_ris_v11/connection.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "stage": "colab_connection",
  "gpu": "Tesla T4",
  "vram_gib": 14.56317138671875,
  "torch": "2.11.0+cu130",
  "python": "3.13.15",
  "training_started": false
}


In [1]:
# V13 fresh GPU verification only; no model training or Drive mounting.
import json, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'No GPU allocated'
x=torch.ones((256,256),device='cuda'); y=x@x; torch.cuda.synchronize()
assert y[0,0].item()==256
r={'checked_at_utc':datetime.now(timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0),'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30,'compute_capability':list(torch.cuda.get_device_capability(0)),'bf16_native':torch.cuda.is_bf16_supported(including_emulation=False),'torch':torch.__version__,'python':platform.python_version(),'cuda_matrix_test':'passed','training_started':False}
Path('/content/v13_gpu_connection.json').write_text(json.dumps(r,indent=2))
print(json.dumps(r,indent=2))
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used','--format=csv,noheader'],text=True))
del x,y
torch.cuda.empty_cache()

{
  "checked_at_utc": "2026-10-05T06:34:46.943674+00:00",
  "gpu": "Tesla T4",
  "vram_gib": 14.56317138671875,
  "compute_capability": [
    7,
    5
  ],
  "bf16_native": false,
  "torch": "2.11.0+cu130",
  "python": "3.13.15",
  "cuda_matrix_test": "passed",
  "training_started": false
}
Tesla T4, 15360 MiB, 153 MiB



In [2]:
# V13 bounded cloud replication. No new holdout evaluation.
import sys, subprocess, requests, hashlib, zipfile, io, json, time, shutil
from pathlib import Path
from IPython.display import display, FileLink
root=Path('/content/v13_cloud_engineering'); root.mkdir(exist_ok=True)
assert not (root/'LAUNCHED').exists(), 'Already launched: inspect existing results, do not duplicate'
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers==0.35.2','transformers==4.57.1','peft==0.17.1','accelerate==1.11.0'],check=True)
url='https://github.com/Maxnopnop/elec4240-marigold-depth/releases/download/v13-cloud-engineering-2026-10-05/cloud_training_probe.zip'
response=requests.get(url,timeout=180); response.raise_for_status()
assert hashlib.sha256(response.content).hexdigest()=='2699b8f72804aeda29bcbf2bdff599e461bba65361c1b073a918c2bfc65454ce'
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    assert all('/' not in n and '\\' not in n for n in z.namelist())
    z.extractall(root)
(root/'LAUNCHED').write_text(time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
print('Starting bounded training-only test:3 arms x64 updates;2h cap;no holdout.',flush=True)
with (root/'run.log').open('w') as log:
    process=subprocess.run([sys.executable,'-u',str(root/'cloud_probe.py')],stdout=log,stderr=subprocess.STDOUT,timeout=7500)
print('Exit code:',process.returncode)
for name in ['environment.json','results.json','completion.json','error.json']:
    p=root/'results'/name
    if p.exists(): print(name,p.read_text())
print((root/'run.log').read_text()[-5000:])
archive=shutil.make_archive('/content/v13_cloud_engineering_results','zip',str(root),'results')
display(FileLink(archive))

Starting bounded training-only test:3 arms x64 updates;2h cap;no holdout.
Exit code: 0
environment.json {
  "gpu": "Tesla T4",
  "capability": [
    7,
    5
  ],
  "native_bf16": false,
  "python": "3.13.15",
  "packages": {
    "torch": "2.11.0+cu130",
    "diffusers": "0.35.2",
    "transformers": "4.57.1",
    "peft": "0.17.1",
    "accelerate": "1.11.0"
  },
  "input_manifest_sha256": "472f7f340fb80b59a9915cdfb2ff5d52dacc1da1d482c21090b2f39522dfe6c8"
}
results.json [
  {
    "arm": "pixel",
    "updates": 64,
    "warm_mean_seconds": 2.323987743124983,
    "peak_mib": 3295.7685546875,
    "resume_exact": true,
    "resume_max_parameter_difference": 0.0,
    "same_dtype_as_local": true
  },
  {
    "arm": "pair_always",
    "updates": 64,
    "warm_mean_seconds": 2.3173361346875083,
    "peak_mib": 3296.6826171875,
    "resume_exact": true,
    "resume_max_parameter_difference": 0.0,
    "same_dtype_as_local": true
  },
  {
    "arm": "pair_ready",
    "updates": 64,
    "warm_mean

/content/v13_cloud_engineering_results.zip

In [3]:
# Archive completed cloud experiment, including logs and all checkpoints.
from pathlib import Path
import json, hashlib, zipfile
from google.colab import files
root=Path('/content/v13_cloud_engineering')
paths=sorted(p for p in root.rglob('*') if p.is_file() and '__pycache__' not in p.parts and p.name not in ['training_only_inputs.pt','initial.pt'])
manifest={str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
archive=Path('/content/v13_cloud_complete_2026-10-05.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in paths:z.write(p,str(p.relative_to(root)))
    z.writestr('ARCHIVE_MANIFEST.json',json.dumps(manifest,indent=2))
print(json.dumps({'archive':str(archive),'bytes':archive.stat().st_size,'sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'files':len(paths)},indent=2))
files.download(str(archive))

{
  "archive": "/content/v13_cloud_complete_2026-10-05.zip",
  "bytes": 37641335,
  "sha256": "43bb0764a0bf318c37a9bc8f3f2a86fd013c123054d65b951664c8e57e8831ae",
  "files": 18
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Engineering probe only: zero optimizer updates; not RIS accuracy.
import subprocess, sys, os, json, time, hashlib
from pathlib import Path
os.environ['USE_TF']='0'
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers==0.35.2','transformers==4.57.1','peft==0.17.1','accelerate==1.11.0'],check=True)
import torch
from diffusers import MarigoldDepthPipeline
from peft import LoraConfig
from importlib.metadata import version
assert torch.cuda.is_available()
torch.manual_seed(17)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
revision='9571e7123e258cf052b4e54241f17971c290e9a8'
start=time.perf_counter()
pipe=MarigoldDepthPipeline.from_pretrained('prs-eth/marigold-depth-v1-1',revision=revision,variant='fp16',torch_dtype=dtype).to('cuda')
pipe.vae.requires_grad_(False); pipe.unet.requires_grad_(False); pipe.text_encoder.requires_grad_(False)
pipe.unet.add_adapter(LoraConfig(r=4,lora_alpha=4,init_lora_weights='gaussian',target_modules=['to_q','to_k','to_v','to_out.0']))
params=[p for p in pipe.unet.parameters() if p.requires_grad]
for p in params: p.data=p.data.float()
pipe.unet.enable_gradient_checkpointing()
pipe.scheduler.set_timesteps(1,device='cuda')
assert pipe.scheduler.timesteps.tolist()==[999] and float(pipe.scheduler.alphas_cumprod[999])==0.
assert pipe.scheduler.config.prediction_type=='v_prediction'
torch.cuda.reset_peak_memory_stats()
rgb=torch.full((1,3,192,256),-.7,device='cuda')
rgb[:,:,48:144,24:104]=torch.tensor([1.,-1.,-1.],device='cuda')[None,:,None,None]
rgb[:,:,48:144,152:232]=torch.tensor([-1.,-1.,1.],device='cuda')[None,:,None,None]
mask=torch.full_like(rgb,-1.); mask[:,:,48:144,24:104]=1.
prompts=['A binary segmentation mask of the red rectangle, white foreground and black background.','A binary segmentation mask of the blue rectangle, white foreground and black background.']
pipe.unet.eval()
with torch.no_grad():
    ids=pipe.tokenizer(prompts,padding='max_length',max_length=77,truncation=True,return_tensors='pt').input_ids.to('cuda')
    embeddings=pipe.text_encoder(ids)[0]
    zrgb=pipe.vae.encode(rgb.to(dtype)).latent_dist.mode()*pipe.vae.config.scaling_factor
    target=pipe.vae.encode(mask.to(dtype)).latent_dist.mode()*pipe.vae.config.scaling_factor
    recon=pipe.vae.decode(target/pipe.vae.config.scaling_factor).sample.float().mean(1)>0
    truth=mask.mean(1)>0
    noise=torch.randn(zrgb.shape,device='cuda',dtype=dtype,generator=torch.Generator(device='cuda').manual_seed(91001))
    inp=torch.cat([zrgb,noise],1)
    a=pipe.unet(inp,999,embeddings[:1]).sample.float()
    b=pipe.unet(inp,999,embeddings[1:]).sample.float()
pipe.unet.train()
with torch.autocast('cuda',dtype=dtype,enabled=(dtype!=torch.float32)):
    loss=torch.nn.functional.mse_loss(pipe.unet(inp,999,embeddings[:1]).sample.float(),-target.float())
loss.backward()
grads=[p.grad for p in params if p.grad is not None]
finite=all(torch.isfinite(g).all().item() for g in grads)
norm=sum(g.float().square().sum().item() for g in grads)**.5
assert finite and norm>0
torch.cuda.synchronize()
report={'status':'engineering_probe_complete','gpu':torch.cuda.get_device_name(),'dtype':str(dtype),'model_revision':revision,'trainable_parameters':sum(p.numel() for p in params),'prompt_output_mae':(a-b).abs().mean().item(),'mask_vae_roundtrip_iou':((recon&truth).sum()/(recon|truth).sum()).item(),'roundtrip_is_not_prediction':True,'loss':loss.item(),'gradient_norm':norm,'finite_gradients':finite,'peak_allocated_mib':torch.cuda.max_memory_allocated()/2**20,'seconds_including_download':time.perf_counter()-start,'optimizer_updates':0,'ris_accuracy_validated':False,'selector_validated':False,'packages':{n:version(n) for n in ['torch','diffusers','transformers','peft','accelerate']}}
Path('/content/generative_ris_v11').mkdir(exist_ok=True)
Path('/content/generative_ris_v11/cloud_preflight.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
del pipe,params,grads,a,b,loss
import gc
gc.collect(); torch.cuda.empty_cache()


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/824 [00:00<?, ?B/s]

...

text_encoder/model.fp16.safetensors:   0%|          | 0.00/681M [00:00<?, ?B/s]

vae/diffusion_pytorch_model.fp16.safeten(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


{
  "status": "engineering_probe_complete",
  "gpu": "Tesla T4",
  "dtype": "torch.bfloat16",
  "model_revision": "9571e7123e258cf052b4e54241f17971c290e9a8",
  "trainable_parameters": 829952,
  "prompt_output_mae": 0.05110782012343407,
  "mask_vae_roundtrip_iou": 1.0,
  "roundtrip_is_not_prediction": true,
  "loss": 2.3882601261138916,
  "gradient_norm": 43.7621241916866,
  "finite_gradients": true,
  "peak_allocated_mib": 2773.94189453125,
  "seconds_including_download": 37.177383719000034,
  "optimizer_updates": 0,
  "ris_accuracy_validated": false,
  "selector_validated": false,
  "packages": {
    "torch": "2.11.0+cu130",
    "diffusers": "0.35.2",
    "transformers": "4.57.1",
    "peft": "0.17.1",
    "accelerate": "1.11.0"
  }
}


In [ ]:
# Save source and results; this is not a trained segmentation model.
import json,hashlib,shutil
from pathlib import Path
from google.colab import files
from IPython.display import display,HTML
root=Path('/content/generative_ris_v11')
source="# Engineering probe only: zero optimizer updates; not RIS accuracy.\nimport subprocess, sys, os, json, time, hashlib\nfrom pathlib import Path\nos.environ['USE_TF']='0'\nsubprocess.run([sys.executable,'-m','pip','install','-q','diffusers==0.35.2','transformers==4.57.1','peft==0.17.1','accelerate==1.11.0'],check=True)\nimport torch\nfrom diffusers import MarigoldDepthPipeline\nfrom peft import LoraConfig\nfrom importlib.metadata import version\nassert torch.cuda.is_available()\ntorch.manual_seed(17)\ndtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32\nrevision='9571e7123e258cf052b4e54241f17971c290e9a8'\nstart=time.perf_counter()\npipe=MarigoldDepthPipeline.from_pretrained('prs-eth/marigold-depth-v1-1',revision=revision,variant='fp16',torch_dtype=dtype).to('cuda')\npipe.vae.requires_grad_(False); pipe.unet.requires_grad_(False); pipe.text_encoder.requires_grad_(False)\npipe.unet.add_adapter(LoraConfig(r=4,lora_alpha=4,init_lora_weights='gaussian',target_modules=['to_q','to_k','to_v','to_out.0']))\nparams=[p for p in pipe.unet.parameters() if p.requires_grad]\nfor p in params: p.data=p.data.float()\npipe.unet.enable_gradient_checkpointing()\npipe.scheduler.set_timesteps(1,device='cuda')\nassert pipe.scheduler.timesteps.tolist()==[999] and float(pipe.scheduler.alphas_cumprod[999])==0.\nassert pipe.scheduler.config.prediction_type=='v_prediction'\ntorch.cuda.reset_peak_memory_stats()\nrgb=torch.full((1,3,192,256),-.7,device='cuda')\nrgb[:,:,48:144,24:104]=torch.tensor([1.,-1.,-1.],device='cuda')[None,:,None,None]\nrgb[:,:,48:144,152:232]=torch.tensor([-1.,-1.,1.],device='cuda')[None,:,None,None]\nmask=torch.full_like(rgb,-1.); mask[:,:,48:144,24:104]=1.\nprompts=['A binary segmentation mask of the red rectangle, white foreground and black background.','A binary segmentation mask of the blue rectangle, white foreground and black background.']\npipe.unet.eval()\nwith torch.no_grad():\n    ids=pipe.tokenizer(prompts,padding='max_length',max_length=77,truncation=True,return_tensors='pt').input_ids.to('cuda')\n    embeddings=pipe.text_encoder(ids)[0]\n    zrgb=pipe.vae.encode(rgb.to(dtype)).latent_dist.mode()*pipe.vae.config.scaling_factor\n    target=pipe.vae.encode(mask.to(dtype)).latent_dist.mode()*pipe.vae.config.scaling_factor\n    recon=pipe.vae.decode(target/pipe.vae.config.scaling_factor).sample.float().mean(1)>0\n    truth=mask.mean(1)>0\n    noise=torch.randn(zrgb.shape,device='cuda',dtype=dtype,generator=torch.Generator(device='cuda').manual_seed(91001))\n    inp=torch.cat([zrgb,noise],1)\n    a=pipe.unet(inp,999,embeddings[:1]).sample.float()\n    b=pipe.unet(inp,999,embeddings[1:]).sample.float()\npipe.unet.train()\nwith torch.autocast('cuda',dtype=dtype,enabled=(dtype!=torch.float32)):\n    loss=torch.nn.functional.mse_loss(pipe.unet(inp,999,embeddings[:1]).sample.float(),-target.float())\nloss.backward()\ngrads=[p.grad for p in params if p.grad is not None]\nfinite=all(torch.isfinite(g).all().item() for g in grads)\nnorm=sum(g.float().square().sum().item() for g in grads)**.5\nassert finite and norm>0\ntorch.cuda.synchronize()\nreport={'status':'engineering_probe_complete','gpu':torch.cuda.get_device_name(),'dtype':str(dtype),'model_revision':revision,'trainable_parameters':sum(p.numel() for p in params),'prompt_output_mae':(a-b).abs().mean().item(),'mask_vae_roundtrip_iou':((recon&truth).sum()/(recon|truth).sum()).item(),'roundtrip_is_not_prediction':True,'loss':loss.item(),'gradient_norm':norm,'finite_gradients':finite,'peak_allocated_mib':torch.cuda.max_memory_allocated()/2**20,'seconds_including_download':time.perf_counter()-start,'optimizer_updates':0,'ris_accuracy_validated':False,'selector_validated':False,'packages':{n:version(n) for n in ['torch','diffusers','transformers','peft','accelerate']}}\nPath('/content/generative_ris_v11').mkdir(exist_ok=True)\nPath('/content/generative_ris_v11/cloud_preflight.json').write_text(json.dumps(report,indent=2))\nprint(json.dumps(report,indent=2))\ndel pipe,params,grads,a,b,loss\nimport gc\ngc.collect(); torch.cuda.empty_cache()\n"
(root/'executed_cloud_probe.py').write_text(source)
manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in root.iterdir() if p.is_file() and p.name!='manifest.json'}
(root/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive('/content/generative_ris_v11_colab_results','zip',str(root))
display(HTML('<h2>Colab engineering check PASSED</h2><p>Tesla T4 | 14.56 GiB available | 829,952 LoRA parameters | finite gradients | peak 2.71 GiB</p><p>Image-generation-derived Marigold: RGB + referring text to mask-target loss.</p><p><b>Zero training updates. RIS accuracy and budget policy are NOT validated.</b></p>'))
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>